In [1]:
import requests 
import logging 
from dotenv import load_dotenv
import os
import ccxt
import pandas as pd
import numpy as np
from numpy import nan as npNan
import pandas_ta as ta
from datetime import datetime, timedelta
import time
import re

load_dotenv()




True

In [2]:
class DataManager:
    
    def __init__(self):
        self.glassnode_api_key = os.getenv('GLASSNODE_API_KEY')
        logging.basicConfig(level=logging.INFO)

    @staticmethod
    def datetime_to_unix(dt_str, dt_format="%Y-%m-%d"):
        """
        Convert a datetime string to a Unix timestamp.
        
        :param dt_str: The datetime string to convert.
        :param dt_format: The format of the datetime string.
        :return: Unix timestamp as an integer.
        """
        dt = datetime.strptime(dt_str, dt_format)
        return int(dt.timestamp())


    def _fetch_glassnode_data(self, endpoint, start, end, frequency):
        params = {
            'a': 'BTC',
            's': self.datetime_to_unix(start),
            'u': self.datetime_to_unix(end),
            'i': frequency,
            'f': 'JSON',
            'api_key': self.glassnode_api_key  
        }
        
        base_url = 'https://api.glassnode.com/v1/metrics'
        url = f"{base_url}/{endpoint}"
        name = re.search(r'/([^/]*)$', endpoint).group(1)

        for attempt in range(3):
            try:
                response = requests.get(url, params=params)
                if response.status_code == 200:
                    data = response.json()
                    df = pd.DataFrame(data)
                    if 't' in df.columns:
                        df['t'] = pd.to_datetime(df['t'], unit='s')
                    df.rename(columns={'v': name}, inplace=True)
                    logging.info(f"Successfully fetched data for endpoint: {endpoint}")
                    return df
                else:
                    logging.error(f"Failed to fetch data: {response.status_code} - {response.text}")
            except Exception as e:
                logging.error(f"Attempt {attempt + 1}: Exception occurred while fetching data: {e}")

        raise Exception(f"Failed to fetch data after 3 attempts for endpoint: {endpoint}")
    
    def _fetch_and_merge_glassnode_data(self, endpoints, start, end, frequency):
        data_frames = []

        for endpoint in endpoints:
            try:
                df = self._fetch_glassnode_data(endpoint, start, end, frequency)
                if df is not None and not df.empty:
                    data_frames.append(df.set_index('t'))
                else:
                    logging.warning(f"No data found for endpoint: {endpoint}")
            except Exception as e:
                logging.error(f"Failed to fetch data for endpoint: {endpoint} with error: {e}")

        if data_frames:
            merged_df = pd.concat(data_frames, axis=1, join='outer')
            merged_df.reset_index(inplace=True)
            merged_df.set_index('t', inplace=True)
            logging.info("Successfully merged data from all endpoints.")
            return merged_df
        else:
            logging.warning("No data frames to merge.")
            return None
    
    def get_trigger_data(self, start, end, frequency='1h'):
        short_term_endpoints = [
            os.getenv('BTC_PRICE'),
            os.getenv('SSR'),
            os.getenv('CVD'),
            os.getenv('SUPPLY_IN_PROFIT'),
            os.getenv('BTC_HASH_RATE')
        ]
        return self._fetch_and_merge_glassnode_data(short_term_endpoints, start, end, frequency)

    def get_context_data(self, start, end, frequency='24h'):
        contextual_endpoints = [
            os.getenv('BTC_PRICE'),
            os.getenv('BTC_REALIZED_PRICE'),
            os.getenv('PUELL_MULTIPLE'),
            os.getenv('MVRV_Z_SCORE'),
            os.getenv('ENTITY_ADJ_NUPL'),
            os.getenv('ENTITY_ADJ_DORMANCY_FLOW'),
            os.getenv('SUPPLY_IN_PROFIT')
        ]
        return self._fetch_and_merge_glassnode_data(contextual_endpoints, start, end, frequency)

    def get_ccxt_data(self, 
                      exchange_id='binance', 
                      symbol='BTC/USDT', 
                      timeframe='1h', 
                      start_time=None, 
                      end_time=None):
        """
        Generic method to retrieve OHLCV data using CCXT from any exchange.
        
        :param exchange_id:     e.g. 'binance', 'bybit', 'coinbasepro', ...
        :param symbol:          e.g. 'BTC/USDT', 'ETH/USDT'
        :param timeframe:       e.g. '1m', '5m', '1h', '1d' etc. (depends on the exchange's supported intervals)
        :param start_time:      Python datetime for start
        :param end_time:        Python datetime for end
        :return:                pandas DataFrame with columns: [open, high, low, close, volume]
        """
        try:
            # Dynamically create the Exchange class
            exchange_class = getattr(ccxt, exchange_id)
            exchange = exchange_class({
                # Enable rate-limit handling (important for large historical fetches)
                'enableRateLimit': True
            })

            since = self.datetime_to_unix(start_time) * 1000 if start_time else None
            end_ts = self.datetime_to_unix(end_time) * 1000 if end_time else None

            all_data = []
            limit = 1000

            while True:
                # Fetch the next batch of OHLCV
                ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=limit)
                if not ohlcv:
                    break

                all_data += ohlcv

                # If we got fewer than 'limit' candles, probably done
                if len(ohlcv) < limit:
                    break

                # Next 'since' starts from the last candle's timestamp
                last_ts = ohlcv[-1][0]
                # Avoid infinite loops if no progress
                if since is not None and last_ts == since:
                    break
                since = last_ts

                # If there's an end_time, stop once we cross it
                if end_ts and last_ts >= end_ts:
                    break

                # Be a bit nice to the exchange
                time.sleep(exchange.rateLimit / 1000)

            if not all_data:
                logging.warning(f"No data returned from {exchange_id} for {symbol}.")
                return None

            # Turn the raw data into a DataFrame
            df = pd.DataFrame(all_data, columns=["timestamp","open","high","low","close","volume"])
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit='ms')
            df.set_index("timestamp", inplace=True)
            df = df.sort_index()

            # If an end_time is specified, filter out data beyond that
            if end_ts:
                df = df[df.index <= pd.to_datetime(end_ts, unit='ms')]

            # Convert numeric columns to float
            for col in ["open", "high", "low", "close", "volume"]:
                df[col] = df[col].astype(float)

            return df

        except Exception as e:
            logging.error(f"Error fetching CCXT data from {exchange_id}: {e}")
            return None

    def compute_triggers(self, start, end):
        start_dt = datetime.strptime(start, '%Y-%m-%d')
        end_dt = datetime.strptime(end, '%Y-%m-%d')
        
        # Retrieve trigger data (Glassnode)
        trigger_data = self.get_trigger_data((start_dt - timedelta(hours=8640)), end_dt)
        
        # Retrieve CCXT data (instead of Bybit/Binance)
        ccxt_data = self.get_ccxt_data(
            exchange_id='binance',      # or 'bybit', 'kraken', etc.
            symbol='BTC/USDT',
            timeframe='1h',
            start_time=(start_dt - timedelta(hours=8640/2)),
            end_time=end_dt
        )
        
        if ccxt_data is not None and trigger_data is not None:
            # Merge CCXT data with trigger data
            trigger_data = trigger_data.merge(ccxt_data, left_index=True, right_index=True, how='outer')

        # Example: some calculations on trigger_data
        # Make sure these columns exist in your Glassnode data:
        trigger_data['rsi_ssr_smoothed'] = ta.ema(
            ta.rsi(trigger_data['ssr_oscillator'], timeperiod=336), 
            timeperiod=800
        )
        trigger_data['rsi_ssr_smoothed_median'] = trigger_data['rsi_ssr_smoothed'].rolling(window=240).median()
        trigger_data['srs'] = trigger_data['rsi_ssr_smoothed'] - trigger_data['rsi_ssr_smoothed_median']
        
        trigger_data['hash_30'] = trigger_data['hash_rate_mean'].rolling(window=30).mean()
        trigger_data['hash_60'] = trigger_data['hash_rate_mean'].rolling(window=60).mean()
        trigger_data['hash_ribbon'] = trigger_data['hash_30'] - trigger_data['hash_60']
        trigger_data['cvd_ema24'] = ta.ema(trigger_data['spot_cvd_sum'], timeperiod=24)

        # Clean up and filter data
        trigger_data.drop(
            columns=[
                'ssr_oscillator','profit_relative','price_usd_close','rsi_ssr_smoothed',
                'rsi_ssr_smoothed_median','hash_rate_mean','hash_30','hash_60','spot_cvd_sum'
            ],
            errors='ignore',
            inplace=True
        )
        
        trigger_data = trigger_data[trigger_data.index >= start_dt]
        
        return trigger_data    
    
    @staticmethod
    def determine_context(row):
        if row['bottom_detection'] != 0 and (row['top_detection'] == 0 or row.name < row.index[row['top_detection'] != 0].max()):
            return int(1)
        elif row['top_detection'] != 0 and (row['bottom_detection'] == 0 or row.name < row.index[row['bottom_detection'] != 0].max()):
            return int(0)
        else:
            return np.nan

    def compute_context(self, start, end):
        start_dt = datetime.strptime(start, '%Y-%m-%d')
        end_dt = datetime.strptime(end, '%Y-%m-%d')
        
        # Retrieve context data
        context_data = self.get_context_data(pd.to_datetime('2011-08-1'), end_dt)
        if context_data is None:
            return None

        gradient_numerator = (context_data['price_usd_close'].diff(28) - context_data['price_realized_usd'].diff(28))
        gradient_mean = gradient_numerator.expanding().mean()
        gradient_std = gradient_numerator.expanding().std()
        context_data['28d_mkt_gradient'] = (gradient_numerator - gradient_mean) / gradient_std

        context_data['mayer_multiple'] = context_data['price_usd_close'] / context_data['price_usd_close'].rolling(200).mean()
        context_data['price_profit_corr'] = context_data['price_usd_close'].rolling(7).corr(context_data['profit_relative'])

        # Simple linear scaling function
        def linear_scale(value, low, high):
            return np.clip((value - low) / (high - low), 0, 1)

        # Normalize metrics to 0-1
        mvrv_norm = linear_scale(context_data['mvrv_z_score'], 0.0, 3.8)
        mayer_norm = linear_scale(context_data['mayer_multiple'], 1.0, 1.3)
        nupl_norm = linear_scale(context_data['net_unrealized_profit_loss_account_based'], 0.0, 0.6)
        gradient_norm = linear_scale(context_data['28d_mkt_gradient'], 0.0, 7.0)

        # Combine into one continuous context measure
        context_data['context'] = (mvrv_norm + mayer_norm + nupl_norm + gradient_norm) / 4.0

        # Drop unnecessary columns
        context_data.drop(columns=['price_usd_close'], inplace=True, errors='ignore')

        context_data = context_data[context_data.index >= start_dt]
        context_data = context_data[context_data.index <= end_dt]

        return context_data
    
    def compute_context_boolean(self, start, end):
        start_dt = datetime.strptime(start, '%Y-%m-%d')
        end_dt = datetime.strptime(end, '%Y-%m-%d')
        
        context_data = self.get_context_data(pd.to_datetime('2011-08-1'), end_dt)
        if context_data is None:
            return None

        gradient_numerator = context_data['price_usd_close'].diff(28) - context_data['price_realized_usd'].diff(28)
        exp_mean = gradient_numerator.expanding().mean()
        exp_std = gradient_numerator.expanding().std()
        context_data['28d_mkt_gradient'] = (gradient_numerator - exp_mean) / exp_std
        
        context_data['mayer_multiple'] = context_data['price_usd_close'] / context_data['price_usd_close'].rolling(200).mean()
        context_data['price_profit_corr'] = context_data['price_usd_close'].rolling(7).corr(context_data['profit_relative'])

        # Detect market tops and bottoms
        context_data['top_detection'] = (
            np.where(context_data['mvrv_z_score'] > 3.8, 1, 0) * 
            np.where(context_data['mayer_multiple'] >= 1.3, 1, 0) *
            np.where(context_data['net_unrealized_profit_loss_account_based'] >= 0.6, 1, 0) *
            np.where(context_data['28d_mkt_gradient'] >= 7, 1, 0)
        ) * context_data['price_usd_close']
        
        context_data['bottom_detection'] = (
            np.where(context_data['mvrv_z_score'] <= 0, 1, 0) * 
            np.where(context_data['mayer_multiple'] <= 0.8, 1, 0) *
            np.where(context_data['price_usd_close'] <= context_data['price_realized_usd'], 1, 0) *
            np.where(context_data['net_unrealized_profit_loss_account_based'] <= 0, 1, 0) *
            np.where(context_data['puell_multiple'] <= 0.5, 1, 0) *
            np.where(context_data['dormancy_flow'] <= 200000, 1, 0)
        ) * context_data['price_usd_close']

        context_data['context'] = context_data.apply(self.determine_context, axis=1)
        context_data['context'].fillna(method='ffill', inplace=True)

        context_data.drop(columns=['top_detection', 'bottom_detection', 'price_usd_close'], inplace=True, errors='ignore')
        context_data = context_data[context_data.index >= start_dt]

        return context_data

    def get_data(self, start, end, contextualize=True, boolean=False):
        # Trigger data
        trigger_data = self.compute_triggers(start=start, end=end)

        # Context data
        if boolean:
            context_data = self.compute_context_boolean(start=start, end=end)
        else:
            context_data = self.compute_context(start=start, end=end)

        if trigger_data is None or context_data is None:
            return None

        trigger_data.reset_index(inplace=True)
        context_data.reset_index(inplace=True)
        context_data['t'] = context_data['t'].dt.tz_localize(None)  # remove tz info if present
        
        # Merge the datasets
        if contextualize:
            full_data = pd.merge_asof(trigger_data, context_data[['t', 'context']], on='t', direction='forward')
        else:
            full_data = pd.merge_asof(trigger_data, context_data, on='t', direction='forward')
            if 'context' in full_data.columns:
                full_data.drop(columns='context', inplace=True)
        
        full_data.ffill(inplace=True)
        full_data.set_index('t', inplace=True)
   
        return full_data


In [3]:
mgmt = DataManager()

In [10]:
mgmt.compute_triggers(start='2025-01-01', end='2025-01-20')

ERROR:root:Failed to fetch data for endpoint: market/price_usd_close with error: strptime() argument 1 must be str, not datetime.datetime
ERROR:root:Failed to fetch data for endpoint: indicators/ssr_oscillator with error: strptime() argument 1 must be str, not datetime.datetime
ERROR:root:Failed to fetch data for endpoint: market/spot_cvd_sum with error: strptime() argument 1 must be str, not datetime.datetime
ERROR:root:Failed to fetch data for endpoint: supply/profit_relative with error: strptime() argument 1 must be str, not datetime.datetime
ERROR:root:Failed to fetch data for endpoint: mining/hash_rate_mean with error: strptime() argument 1 must be str, not datetime.datetime
ERROR:root:Error fetching CCXT data from binance: strptime() argument 1 must be str, not datetime.datetime


TypeError: 'NoneType' object is not subscriptable